Install and Import packages

In [1]:
pip install sqlalchemy psycopg2-binary

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
import pandas as pd

In [4]:
from sqlalchemy import create_engine

In [5]:
from datetime import datetime

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [7]:
import re

In [8]:
import os

In [9]:
from dotenv import load_dotenv

Extract Layer

In [10]:
roles = [
    "data analyst",
    "data scientist",
    "data engineer",
    "business analyst",
    "machine learning engineer",
    "analytics engineer",
    "data specialist",
    "research analyst"
    "business intelligence analyst",
    "data architect",
    "data warehouse engineer"
]

In [11]:

url = "https://data.usajobs.gov/api/search"

headers = {
    "User-Agent": "mhaftran@gmail.com",
    "Authorization-Key": "NK21ZnEiSAktOfnXLZwzGYsW2mf/U5ouL3hkCort544="
}

all_jobs = []

for role in roles:


    params = {
            "Keyword": role,
            "ResultsPerPage": 2800,
            "Page": 1,
        }

    response = requests.get(url, headers=headers, params=params)
    data = response.json()

    jobs = data["SearchResult"]["SearchResultItems"]

    for job in jobs:
        obj = job["MatchedObjectDescriptor"]

        all_jobs.append({
            "role": role,


            "title": obj.get("PositionTitle"),
            "location": obj.get("PositionLocationDisplay"),
            "description": obj["UserArea"]["Details"].get("JobSummary"),


            "posted_date": obj.get("PublicationStartDate"),
            "close_date": obj.get("ApplicationCloseDate"),


            "job_grade": obj.get("JobGrade"),


            "salary_min": obj.get("PositionRemuneration", [{}])[0].get("MinimumRange"),
            "salary_max": obj.get("PositionRemuneration", [{}])[0].get("MaximumRange"),
            "salary_interval": obj.get("PositionRemuneration", [{}])[0].get("RateIntervalCode"),


            "employment_type": obj.get("PositionSchedule", [{}])[0] if isinstance(obj.get("PositionSchedule"), list) else obj.get("PositionSchedule"),

        })

In [12]:
df = pd.DataFrame(all_jobs)
print(df.shape)
df.head()

(1249, 11)


,role,title,location,description,posted_date,close_date,job_grade,salary_min,salary_max,salary_interval,employment_type
0,data analyst,Data Analyst,"Arlington, Virginia",The USPS OIG is seeking a highly qualified app...,2026-06-11T11:56:33.5330,2026-06-22T23:59:59.9970,[{'Code': 'GG'}],70623,158322,PA,"{'Name': '', 'Code': '1'}"
1,data analyst,Data Analyst,"Arlington, Virginia",*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11T11:49:11.8070,2026-06-22T23:59:59.9970,[{'Code': 'GG'}],70623,158322,PA,"{'Name': '', 'Code': '1'}"
2,data analyst,Program Analyst (Data Analyst),Multiple Locations,As a Program Analyst (Data Analyst) at the GS-...,2026-06-10T00:00:00.0000,2026-06-16T23:59:59.9970,[{'Code': 'GS'}],63795,99404,PA,"{'Name': '', 'Code': '1'}"
3,data analyst,"Head, Data Analytics and Reporting Office (Sup...","Washington, District of Columbia",This position serves as a Supervisory Program ...,2026-06-12T13:54:33.8430,2026-06-26T23:59:59.9970,[{'Code': 'GS'}],143913,187093,PA,"{'Name': 'Flexitime.', 'Code': '1'}"
4,data analyst,Program Analyst,"Johnson City, Tennessee",The Productivity Program Analyst is a member o...,2026-06-12T10:49:08.6800,2026-06-22T23:59:59.9970,[{'Code': 'GS'}],50460,80243,PA,"{'Name': '', 'Code': '1'}"


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1249 entries, 0 to 1248
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   role             1249 non-null   object
 1   title            1249 non-null   object
 2   location         1249 non-null   object
 3   description      1249 non-null   object
 4   posted_date      1249 non-null   object
 5   close_date       1249 non-null   object
 6   job_grade        1249 non-null   object
 7   salary_min       1249 non-null   object
 8   salary_max       1249 non-null   object
 9   salary_interval  1249 non-null   object
 10  employment_type  1249 non-null   object
dtypes: object(11)
memory usage: 107.5+ KB


**Transform Layer**

Transform Job Grade

In [14]:
df["job_grade"] = df["job_grade"].apply(
    lambda x: x[0]["Code"] if isinstance(x, list) and len(x) > 0 else x
)

Transform Employment Type

In [15]:
df["employment_type"] = df["employment_type"].apply(
    lambda x: x.get("Code") if isinstance(x, dict) else x
)

Change Date Format

In [16]:
df["posted_date"] = pd.to_datetime(df["posted_date"], errors="coerce").dt.date
df["close_date"] = pd.to_datetime(df["close_date"], errors="coerce").dt.date

In [17]:
df

,role,title,location,description,posted_date,close_date,job_grade,salary_min,salary_max,salary_interval,employment_type
0,data analyst,Data Analyst,"Arlington, Virginia",The USPS OIG is seeking a highly qualified app...,2026-06-11,2026-06-22,GG,70623,158322,PA,1
1,data analyst,Data Analyst,"Arlington, Virginia",*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623,158322,PA,1
2,data analyst,Program Analyst (Data Analyst),Multiple Locations,As a Program Analyst (Data Analyst) at the GS-...,2026-06-10,2026-06-16,GS,63795,99404,PA,1
3,data analyst,"Head, Data Analytics and Reporting Office (Sup...","Washington, District of Columbia",This position serves as a Supervisory Program ...,2026-06-12,2026-06-26,GS,143913,187093,PA,1
4,data analyst,Program Analyst,"Johnson City, Tennessee",The Productivity Program Analyst is a member o...,2026-06-12,2026-06-22,GS,50460,80243,PA,1
...,...,...,...,...,...,...,...,...,...,...,...
1244,data architect,CIVIL ENGINEER,Multiple Locations,"Click on ""Learn more about this agency"" button...",2025-12-22,2026-12-21,GS,74678,192331,PA,1
1245,data architect,Healthcare Engineer,"Columbia, South Carolina",This position serves as a Project Engineer at ...,2026-06-03,2026-06-15,GS,114312,148609,PA,1
1246,data architect,SUPERVISORY CONTRACT SPECIALIST (Title 5),"Austin, Texas",This position is located within the USPFO at C...,2026-06-05,2026-06-20,GS,109428,142259,PA,1
1247,data warehouse engineer,IT Specialist (Enterprise Data Platform),"Washington, District of Columbia",The Lead Data Engineer - Enterprise Data Platf...,2026-06-09,2026-06-16,AD,102415,191965,PA,1


Drop Duplicates

In [18]:
df = df.drop_duplicates()

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1223 entries, 0 to 1248
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   role             1223 non-null   object
 1   title            1223 non-null   object
 2   location         1223 non-null   object
 3   description      1223 non-null   object
 4   posted_date      1223 non-null   object
 5   close_date       1223 non-null   object
 6   job_grade        1223 non-null   object
 7   salary_min       1223 non-null   object
 8   salary_max       1223 non-null   object
 9   salary_interval  1223 non-null   object
 10  employment_type  1223 non-null   object
dtypes: object(11)
memory usage: 114.7+ KB


Extract key skills

In [20]:
df_1=df.copy()

In [21]:
skill_filter = [
    "python", "sql", "tableau", "power bi", "excel",
    "aws", "spark", "snowflake", "machine learning"
]

In [22]:
texts = df_1["description"].fillna("").str.lower()

In [23]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=1000)
X = vectorizer.fit_transform(df_1["description"].fillna("").str.lower())

In [24]:
feature_names = vectorizer.get_feature_names_out()

In [25]:
# Get top keywords. I try to give those skill filter a boost
def get_top_keywords(row, top_n=10):
    row = row.toarray().flatten()

    scored_words = []

    for i, score in enumerate(row):
        word = feature_names[i]

        if score > 0:
            # base TF-IDF score
            final_score = score

            # BOOST if it's a skill
            if word in skill_filter:
                final_score *= 3

            scored_words.append((word, final_score))

    # sort by final score
    scored_words.sort(key=lambda x: x[1], reverse=True)

    return [word for word, score in scored_words[:top_n]]

In [26]:
df_1["top_keywords"] = [
    get_top_keywords(X[i]) for i in range(X.shape[0])
]

In [27]:
df_1

,role,title,location,description,posted_date,close_date,job_grade,salary_min,salary_max,salary_interval,employment_type,top_keywords
0,data analyst,Data Analyst,"Arlington, Virginia",The USPS OIG is seeking a highly qualified app...,2026-06-11,2026-06-22,GG,70623,158322,PA,1,"[voice, applicant, skills, virginia, highly, s..."
1,data analyst,Data Analyst,"Arlington, Virginia",*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623,158322,PA,1,"[postal, united, states, service, inspection, ..."
2,data analyst,Program Analyst (Data Analyst),Multiple Locations,As a Program Analyst (Data Analyst) at the GS-...,2026-06-10,2026-06-16,GS,63795,99404,PA,1,"[data, analyst, recommendations, talent, selec..."
3,data analyst,"Head, Data Analytics and Reporting Office (Sup...","Washington, District of Columbia",This position serves as a Supervisory Program ...,2026-06-12,2026-06-26,GS,143913,187093,PA,1,"[library, reporting, financial, collections, l..."
4,data analyst,Program Analyst,"Johnson City, Tennessee",The Productivity Program Analyst is a member o...,2026-06-12,2026-06-22,GS,50460,80243,PA,1,"[data, ambulatory, clinic, contribute, respons..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1244,data architect,CIVIL ENGINEER,Multiple Locations,"Click on ""Learn more about this agency"" button...",2025-12-22,2026-12-21,GS,74678,192331,PA,1,"[button, learn, important, click, additional, ..."
1245,data architect,Healthcare Engineer,"Columbia, South Carolina",This position serves as a Project Engineer at ...,2026-06-03,2026-06-15,GS,114312,148609,PA,1,"[construction, designs, projects, manages, pro..."
1246,data architect,SUPERVISORY CONTRACT SPECIALIST (Title 5),"Austin, Texas",This position is located within the USPFO at C...,2026-06-05,2026-06-20,GS,109428,142259,PA,1,"[specialists, uspfo, services, contracting, ta..."
1247,data warehouse engineer,IT Specialist (Enterprise Data Platform),"Washington, District of Columbia",The Lead Data Engineer - Enterprise Data Platf...,2026-06-09,2026-06-16,AD,102415,191965,PA,1,"[case, management, data, applies, judiciary, c..."


=> This didn't work, as the keywords extracted were not important skills

Mapping method

In [28]:
skills = [
    # programming
    "python", "r", "sql", "java",

    # data tools
    "excel", "tableau", "power bi",

    # cloud / engineering
    "aws", "azure", "snowflake", "spark", "databricks",

    # analytics / ml
    "data analysis", "statistics", "machine learning",
    "regression", "forecasting",

    # business
    "data visualization", "reporting", "etl"
]


In [29]:
'''
df["skills_found"] = df["description"].str.lower().apply(
    lambda text: [
        skill for skill in skills
        if re.search(rf"\b{re.escape(skill)}\b", text) # make the keyword matching more precise
    ]
)
'''

'\ndf["skills_found"] = df["description"].str.lower().apply(\n    lambda text: [\n        skill for skill in skills\n        if re.search(rf"\x08{re.escape(skill)}\x08", text) # make the keyword matching more precise\n    ]\n)\n'

In [30]:
'''
df["skills_found"] = df["skills_found"].apply(
    lambda x: x if isinstance(x, list) else []
)
'''

'\ndf["skills_found"] = df["skills_found"].apply(\n    lambda x: x if isinstance(x, list) else []\n)\n'

**Load**

In [31]:
df.to_csv("job_skills.csv", index=False)

In [32]:
df.columns = df.columns.str.upper().str.strip()
df = df.reset_index(drop=True)
df = df.rename(columns={"ROLE": "JOB_ROLE"})


In [33]:
df

,JOB_ROLE,TITLE,LOCATION,DESCRIPTION,POSTED_DATE,CLOSE_DATE,JOB_GRADE,SALARY_MIN,SALARY_MAX,SALARY_INTERVAL,EMPLOYMENT_TYPE
0,data analyst,Data Analyst,"Arlington, Virginia",The USPS OIG is seeking a highly qualified app...,2026-06-11,2026-06-22,GG,70623,158322,PA,1
1,data analyst,Data Analyst,"Arlington, Virginia",*PLEASE NOTE THAT THIS VACANCY ANNOUNCEMENT IS...,2026-06-11,2026-06-22,GG,70623,158322,PA,1
2,data analyst,Program Analyst (Data Analyst),Multiple Locations,As a Program Analyst (Data Analyst) at the GS-...,2026-06-10,2026-06-16,GS,63795,99404,PA,1
3,data analyst,"Head, Data Analytics and Reporting Office (Sup...","Washington, District of Columbia",This position serves as a Supervisory Program ...,2026-06-12,2026-06-26,GS,143913,187093,PA,1
4,data analyst,Program Analyst,"Johnson City, Tennessee",The Productivity Program Analyst is a member o...,2026-06-12,2026-06-22,GS,50460,80243,PA,1
...,...,...,...,...,...,...,...,...,...,...,...
1218,data architect,CIVIL ENGINEER,Multiple Locations,"Click on ""Learn more about this agency"" button...",2025-12-22,2026-12-21,GS,74678,192331,PA,1
1219,data architect,Healthcare Engineer,"Columbia, South Carolina",This position serves as a Project Engineer at ...,2026-06-03,2026-06-15,GS,114312,148609,PA,1
1220,data architect,SUPERVISORY CONTRACT SPECIALIST (Title 5),"Austin, Texas",This position is located within the USPFO at C...,2026-06-05,2026-06-20,GS,109428,142259,PA,1
1221,data warehouse engineer,IT Specialist (Enterprise Data Platform),"Washington, District of Columbia",The Lead Data Engineer - Enterprise Data Platf...,2026-06-09,2026-06-16,AD,102415,191965,PA,1


In [34]:
load_dotenv()

True

In [35]:
USER = os.getenv("DB_USER")
PASSWORD = os.getenv("DB_PASSWORD")
HOST = os.getenv("DB_HOST")
PORT = os.getenv("DB_PORT")
DATABASE = os.getenv("DB_NAME")

In [36]:
database_url = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}"
engine = create_engine(database_url)

In [37]:
df.to_sql(
    name='job_skills', 
    con=engine, 
    if_exists='replace', 
    index=False, 
    method='multi' 
)

print("Data pipeline complete! The 'job_skills' table is populated in PostgreSQL.")

Data pipeline complete! The 'job_skills' table is populated in PostgreSQL.
